# Pipeline Demo — ecommerce-recsys-mlops

Este notebook mostra o fluxo completo do pipeline implementado até agora:

1. **Carregamento** — `load_dataset()` busca/cacheia o dataset RetailRocket
2. **Pré-processamento** — `build_interactions()` pondera eventos e agrega por par usuário-item
3. **Split temporal** — `temporal_split()` divide em treino/validação/teste sem vazar o futuro
4. **Modelo (PopularityRecommender)** — baseline via `RecommenderFactory`
5. **Métricas de ranking** — Precision@K, Recall@K, NDCG@K, Hit Rate@K
6. **Métricas de negócio** — Coverage e Revenue@K

> **Pré-requisito**: `make requirements` ou `uv sync` antes de executar.

## 0. Imports e configuração

In [3]:
from pathlib import Path
import sys

# Garante que o pacote src/ é encontrado ao rodar o notebook da raiz do projeto
ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import get_settings

# get_settings() lê o .env e configura logging — chamar uma vez no topo
cfg = get_settings()

print(f"RANDOM_SEED      = {cfg.RANDOM_SEED}")
print(f"TEST_SIZE        = {cfg.TEST_SIZE}")
print(f"VALIDATION_SIZE  = {cfg.VALIDATION_SIZE}")
print(f"RECOMMENDATION_K = {cfg.RECOMMENDATION_K}")

RANDOM_SEED      = 42
TEST_SIZE        = 0.2
VALIDATION_SIZE  = 0.2
RECOMMENDATION_K = 10


## 1. Carregamento do dataset

`load_dataset()` verifica se o arquivo processado existe em `data/processed/`.
- Se existir → lê do cache (rápido).
- Se não existir → baixa via `kagglehub`, consolida e salva o cache.

O schema de saída é: `user_id | item_id | event | value | timestamp`.

In [4]:
from src.data.loader import load_dataset

# force_rebuild=False usa o cache se disponível
events = load_dataset(force_rebuild=False)

print(f"Shape: {events.shape}")
print(f"Eventos únicos: {events['event'].value_counts().to_dict()}")
events.head()

2026-06-30 08:32:24,538 - [INFO    ] - src.data.loader           - loader:129             - request_id=-                                    - Carregando dataset consolidado do cache: /home/gustavodellanhol/_projetos/ecommerce-recsys-mlops/data/processed/dataset_consolidated.csv


Shape: (2494626, 5)
Eventos únicos: {'view': 2404145, 'addtocart': 68499, 'transaction': 21982}


,user_id,item_id,event,value,timestamp
0,257597,355908,view,101520.0,2015-06-02 05:02:12.117
1,992329,248676,view,19440.0,2015-06-02 05:50:14.164
2,483717,253185,view,34740.0,2015-06-02 05:12:35.914
3,951259,367447,view,319800.0,2015-06-02 05:02:17.106
4,972639,22556,view,4320.0,2015-06-02 05:48:06.234


## 2. Pré-processamento — eventos → interações ponderadas

`build_interactions()` aplica o padrão **Strategy** via `WeightedInteractionPreprocessor`:
- Pondera cada evento: `view=1`, `addtocart=2`, `transaction=3`
- Agrega por `(user_id, item_id)`: soma o score, pega o `timestamp` mais recente
- Valida o schema de saída com Pandera antes de retornar

O schema de saída é: `user_id | item_id | score | value | timestamp`.

In [5]:
from src.data.preprocessor import build_interactions

interactions = build_interactions(events)

print(f"Shape: {interactions.shape}")
print(f"Score range: [{interactions['score'].min()}, {interactions['score'].max()}]")
interactions.head()

Shape: (1919316, 5)
Score range: [1, 308]


,user_id,item_id,score,value,timestamp
0,0,67045,1,719880.0,2015-09-11 20:55:17.175
1,0,285930,1,130320.0,2015-09-11 20:49:49.439
2,0,357564,1,254880.0,2015-09-11 20:52:39.591
3,1,72028,1,354960.0,2015-08-13 17:46:06.444
4,2,216305,2,57120.0,2015-08-07 18:17:43.170


### 2a. Alternativa: usando o pipeline sklearn

`build_preprocessing_pipeline()` encapsula a mesma lógica em um `sklearn.Pipeline`,
útil para encadear com transformações futuras ou para uso no `train.py`.

Passa uma estratégia diferente para trocar o comportamento sem mudar o pipeline.

In [6]:
from src.data.preprocessor import WeightedInteractionPreprocessor, build_preprocessing_pipeline

pipeline = build_preprocessing_pipeline(preprocessor=WeightedInteractionPreprocessor())

# transform() retorna o DataFrame de interações (sem validação Pandera — use build_interactions para produção)
interactions_via_pipeline = pipeline.transform(events)
print(
    f"Resultado idêntico ao build_interactions: {interactions_via_pipeline.shape == interactions.shape}"
)

Resultado idêntico ao build_interactions: True


## 3. Split temporal 60/20/20

`temporal_split()` ordena por `timestamp` e corta em três fatias cronológicas.
Nunca usa aleatoriedade — isso impede que interações futuras vazem para o treino.

In [7]:
from src.data.split import temporal_split

train_df, val_df, test_df = temporal_split(
    interactions,
    test_size=cfg.TEST_SIZE,
    validation_size=cfg.VALIDATION_SIZE,
)

print(f"Treino:    {len(train_df):>7,} interações  ({len(train_df)/len(interactions):.0%})")
print(f"Validação: {len(val_df):>7,} interações  ({len(val_df)/len(interactions):.0%})")
print(f"Teste:     {len(test_df):>7,} interações  ({len(test_df)/len(interactions):.0%})")
print()
print(f"Treino cobre até:  {train_df['timestamp'].max()}")
print(f"Validação cobre:   {val_df['timestamp'].min()} → {val_df['timestamp'].max()}")
print(f"Teste cobre a partir de: {test_df['timestamp'].min()}")

2026-06-30 08:32:30,768 - [INFO    ] - src.data.split            - split:36               - request_id=-                                    - Split temporal: 1151589 treino, 383863 validação, 383864 teste, 253353 usuários cold-start no teste


Treino:    1,151,589 interações  (60%)
Validação: 383,863 interações  (20%)
Teste:     383,864 interações  (20%)

Treino cobre até:  2015-07-22 17:14:33.546000
Validação cobre:   2015-07-22 17:14:39.021000 → 2015-08-18 18:43:35.026000
Teste cobre a partir de: 2015-08-18 18:43:35.350000


## 4. Modelo — PopularityRecommender via Factory

`RecommenderFactory.create(name, config)` instancia o modelo correto a partir de uma string.
Isso permite trocar o modelo em `params.yaml` (DVC) sem mudar `train.py`.

Todos os modelos seguem o padrão **Strategy** via `BaseRecommender`: `fit` → `recommend` → `get_params`.

In [8]:
from src.models.factory import RecommenderFactory

model = RecommenderFactory.create("popularity", config={})

# Treina apenas no conjunto de treino — nunca no conjunto completo
model.fit(train_df)

print(f"Modelo: {type(model).__name__}")
print(f"Params: {model.get_params()}")

Modelo: PopularityRecommender
Params: {'model': 'popularity'}


In [9]:
# Exemplo de recomendação para um usuário específico
sample_user = train_df["user_id"].iloc[0]
recs = model.recommend(user_id=sample_user, k=cfg.RECOMMENDATION_K)

print(f"Top-{cfg.RECOMMENDATION_K} recomendações para user_id={sample_user}:")
print(recs)

Top-10 recomendações para user_id=829044:
[5411, 309778, 370653, 461686, 257040, 369447, 298009, 7943, 111530, 335975]


## 5. Métricas de ranking

Avaliamos sobre o conjunto de **teste** (hold-out futuro).
Para cada usuário no teste, os itens com `transaction` são o ground truth.

In [10]:
from src.metrics.ranking import hit_rate_at_k, ndcg_at_k, precision_at_k, recall_at_k

K = cfg.RECOMMENDATION_K

# Usuários presentes no teste E que também estavam no treino
test_users = set(test_df["user_id"]) & set(train_df["user_id"])

# Ground truth: itens de maior relevância (transações) por usuário no teste
test_transactions = test_df[
    test_df["event"].isin(["transaction"])
    if "event" in test_df.columns
    else test_df.index.notnull()
]

# Como interactions já agrega eventos, usamos score >= 3 como proxy de transação
relevant_per_user = (
    test_df[test_df["score"] >= 3].groupby("user_id")["item_id"].apply(set).to_dict()
)

metrics = {"precision": [], "recall": [], "ndcg": [], "hit_rate": []}

for user_id in list(test_users)[:500]:  # limitamos a 500 usuários para ser rápido no demo
    relevant = relevant_per_user.get(user_id, set())
    if not relevant:
        continue
    recs = model.recommend(user_id=user_id, k=K)
    metrics["precision"].append(precision_at_k(recs, relevant, K))
    metrics["recall"].append(recall_at_k(recs, relevant, K))
    metrics["ndcg"].append(ndcg_at_k(recs, relevant, K))
    metrics["hit_rate"].append(hit_rate_at_k(recs, relevant, K))

import statistics

print(f"Avaliação sobre {len(metrics['precision'])} usuários com ≥1 transação no teste:")
print(f"  Precision@{K}  = {statistics.mean(metrics['precision']):.4f}")
print(f"  Recall@{K}     = {statistics.mean(metrics['recall']):.4f}")
print(f"  NDCG@{K}       = {statistics.mean(metrics['ndcg']):.4f}")
print(f"  Hit Rate@{K}   = {statistics.mean(metrics['hit_rate']):.4f}")

Avaliação sobre 73 usuários com ≥1 transação no teste:
  Precision@10  = 0.0014
  Recall@10     = 0.0005
  NDCG@10       = 0.0012
  Hit Rate@10   = 0.0137


## 6. Métricas de negócio

- **Coverage**: que fração do catálogo o modelo recomenda?
- **Revenue@K**: soma do `value` dos itens recomendados que o usuário de fato comprou.

In [11]:
from src.metrics.business import coverage, revenue_at_k

catalog_size = interactions["item_id"].nunique()

# Coletamos todas as listas de recomendação (amostra de 500 usuários)
all_recs = []
revenue_list = []

# Mapa item_id → value no conjunto de teste
item_value_map = test_df.set_index("item_id")["value"].to_dict()

for user_id in list(test_users)[:500]:
    recs = model.recommend(user_id=user_id, k=K)
    all_recs.append(recs)

    # relevant_with_value: apenas itens que o usuário comprou (score ≥ 3)
    user_purchased = test_df[(test_df["user_id"] == user_id) & (test_df["score"] >= 3)]
    relevant_with_value = user_purchased.set_index("item_id")["value"].to_dict()
    revenue_list.append(revenue_at_k(recs, relevant_with_value, K))

print(
    f"Coverage@{K}    = {coverage(all_recs, catalog_size):.4f}  ({coverage(all_recs, catalog_size)*100:.1f}% do catálogo)"
)
print(f"Revenue@{K}     = {sum(revenue_list):.2f}  (soma sobre a amostra de usuários)")
print(f"  → Avg Revenue = {statistics.mean(revenue_list):.4f} por usuário")

Coverage@10    = 0.0001  (0.0% do catálogo)
Revenue@10     = 16440.00  (soma sobre a amostra de usuários)
  → Avg Revenue = 32.8800 por usuário


## Resumo do pipeline

```
load_dataset()                          # data/loader.py  → eventos brutos
    ↓
build_interactions()                    # data/preprocessor.py → score ponderado por par user-item
    ↓
temporal_split(60/20/20)                # data/split.py → train / val / test
    ↓
RecommenderFactory.create("popularity") # models/factory.py + models/popularity.py
    ↓  model.fit(train_df)
    ↓  model.recommend(user_id, k)
    ↓
precision_at_k / recall_at_k            # metrics/ranking.py
ndcg_at_k / hit_rate_at_k
coverage / revenue_at_k                 # metrics/business.py
```

**Próximas etapas (ainda não implementadas):**
- `SVDRecommender` — fatoração de matrizes (scikit-surprise)
- `MLPRecommender` — embeddings em PyTorch
- `src/tracking/mlflow_utils.py` — logging de runs via MLflow
- `src/training/train.py` — script de treino orquestrado pelo DVC